In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns


In [ ]:
df=pd.read_csv('train.txt',sep=';',header=None,names=['text','emotion'])



In [ ]:
df.head()

,text,emotion
0,i didnt feel humiliated,sadness
1,i can go from feeling so hopeless to so damned...,sadness
2,im grabbing a minute to post i feel greedy wrong,anger
3,i am ever feeling nostalgic about the fireplac...,love
4,i am feeling grouchy,anger


In [ ]:
df.isnull().sum()

,0
text,0
emotion,0


In [ ]:
#output variable to numeric terms
unique_emotions=df['emotion'].unique()
emotion_number={}
i=0
for emo in unique_emotions:
    emotion_number[emo]=i
    i+=1
df['emotion']=df['emotion'].map(emotion_number)


In [ ]:
df.head()

,text,emotion
0,i didnt feel humiliated,0
1,i can go from feeling so hopeless to so damned...,0
2,im grabbing a minute to post i feel greedy wrong,1
3,i am ever feeling nostalgic about the fireplac...,2
4,i am feeling grouchy,1


In [ ]:
df['text']=df['text'].apply(lambda x : x.lower())

In [ ]:
#remove punctuation
import string
def remove_punc(text):
  return text.translate(str.maketrans('','',string.punctuation))

In [ ]:
df['text']=df['text'].apply(remove_punc)

In [ ]:
#remove numbers
def remove_numbers(txt):
  new=""
  for i in txt:
    if not i.isdigit():
      new=new+i
  return new
df['text'] =df['text'].apply(remove_numbers)

In [ ]:
#remove rmojis
def remove_emojis(txt):
  new=""
  for i in txt:
    if  i.isascii():
      new=new+i
  return new
df['text'] =df['text'].apply(remove_emojis)

In [ ]:
#remove stopwords
import nltk
nltk.download('stopwords')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
stop_words = set(stopwords.words('english'))

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.


In [ ]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

In [ ]:
df.loc[1]['text']

'i can go from feeling so hopeless to so damned hopeful just from being around someone who cares and is awake'

In [ ]:
#tokenization
def remove(txt):
  words = txt.split()
  cleaned = []
  for i in words:
    if not i in stop_words:
      cleaned.append(i)

  return ' '.join(cleaned)


In [ ]:
df['text'] = df['text'].apply(remove)

In [ ]:
df.loc[1]['text']

'go feeling hopeless damned hopeful around someone cares awake'

In [ ]:
df.head()

,text,emotion
0,didnt feel humiliated,0
1,go feeling hopeless damned hopeful around some...,0
2,im grabbing minute post feel greedy wrong,1
3,ever feeling nostalgic fireplace know still pr...,2
4,feeling grouchy,1


In [ ]:
df.shape


(16000, 2)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(df['text'], df['emotion'], test_size=0.20, random_state=42)

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score

bow_vectorizer = CountVectorizer()
X_train_bow = bow_vectorizer.fit_transform(X_train)
X_test_bow = bow_vectorizer.transform(X_test)


#multinominal naive bayes fir text classifier
nb_model = MultinomialNB()
nb_model.fit(X_train_bow, y_train)


pred_bow = nb_model.predict(X_test_bow)
print(accuracy_score(y_test, pred_bow))



0.768125


In [ ]:
TfidfVectorizer_model = TfidfVectorizer()
x_train_tfidf = TfidfVectorizer_model.fit_transform(X_train)
x_test_tfidf = TfidfVectorizer_model.transform(X_test)

In [ ]:
nb2_model = MultinomialNB()
nb2_model.fit(x_train_tfidf, y_train)

MultinomialNB()

In [ ]:
y_pred=nb2_model.predict(x_test_tfidf)

In [ ]:
print(accuracy_score(y_test, y_pred))

0.6609375


In [ ]:
from sklearn.linear_model import LogisticRegression


In [ ]:
logistic_model = LogisticRegression(max_iter=1000)# specifies the maximum number of iterations that the
# optimization algorithm (solver) will take to converge to a solution during the training process

In [ ]:
logistic_model.fit(x_train_tfidf, y_train)

LogisticRegression(max_iter=1000)

In [ ]:
log_pred = logistic_model.predict(x_test_tfidf)

In [ ]:
print(accuracy_score(y_test, log_pred))

0.8628125


In [ ]:
import pickle

# Save the Logistic Regression model (your best performer)
with open('emotion_model.pkl', 'wb') as f:
    pickle.dump(logistic_model, f)

# Save the TF-IDF Vectorizer
with open('tfidf_vectorizer.pkl', 'wb') as f:
    pickle.dump(TfidfVectorizer_model, f)